# Simulation parameter test

Edit the **Parameters** cell, then run all cells to inspect one simulation, its animation, and its outcomes. After changing parameters, rerun that cell and every cell below it. Keep the seed fixed when comparing settings; change it to inspect another stochastic realization.

The configuration uses the supplied defaults. The animation follows `../section_3/synthetic_data_inference.ipynb`. Set `SMOKE = True` for a short execution check with 2 killers, 3 targets, and a duration of 0.4; this small run is not intended to assess killing behaviour.

Use a Python environment with NumPy, pandas, Matplotlib, and IPython. Run the notebook from anywhere inside this checkout. Spatial and temporal quantities use the model's consistent unit system.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src/mantishrimp").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the MantiShrimp checkout.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

from mantishrimp import (
    CytotoxicityConfig, DomainConfig, MechanicsConfig,
    MotilityConfig, OutputConfig, SimulationConfig, simulate,
)
from mantishrimp.animation import animate

## Parameters

Change `K_BIND`, `K_UNBIND`, and `SEED` below, or edit any field inside `make_config`. The starting binding and unbinding rates, 5.0 and 0.5, match the reference notebook. Seed 42 makes the initial run reproducible.

The periodic domain is explicitly set to the package defaults, from −600 to 600 on each axis. `initial_target_state` is susceptibility; `initial_death_factor` is initial damage. Targets start undamaged. `record_every` controls saved snapshots, while `max_dt` limits integration steps. Animation settings in a later cell only affect playback.

In [ ]:
K_BIND = 5.0
K_UNBIND = 0.5
SEED = 42
SMOKE = False


def make_config(k_bind, k_unbind, seed, smoke=False):
    return SimulationConfig(
        n_killers=2 if smoke else 25,
        n_targets=3 if smoke else 100,
        killer_radius=10, target_radius=12,
        duration=0.4 if smoke else 50,
        max_dt=0.2, max_drift_displacement=1, seed=seed,
        initial_separation_factor=2.5,
        domain=DomainConfig(
            x_min=-600, x_max=600,
            y_min=-600, y_max=600,
            boundary="periodic",
        ),
        mechanics=MechanicsConfig(
            repulsion_kk=50, repulsion_tt=50, repulsion_kt=50,
            adhesion_kt=300, binding_rate=k_bind, unbinding_rate=k_unbind,
        ),
        motility=MotilityConfig(
            killer_speed=50, target_speed=50,
            killer_polarity_decay=0.00001, target_polarity_decay=5.0,
            killer_polarity_noise=0.8, target_polarity_noise=0.2,
            killer_translation_noise=1, target_translation_noise=1,
        ),
        cytotoxicity=CytotoxicityConfig(
            killing_rate_min=0,
            killing_rate_max=1.5,
            death_threshold=1,
            recovery_rate=0,
            kill_probability=1,
            kill_probability_decay_per_synapse=0,
            initial_killer_state=1,
            initial_target_state=1,
            initial_death_factor=0,
            exhaustion_rate_min=0.001, exhaustion_rate_max=0.3,
            instant_kill=False,
        ),
        output=OutputConfig(record_every=1),
    )


config = make_config(K_BIND, K_UNBIND, SEED, smoke=SMOKE)
config.validate()
print(
    f"{config.n_killers} killers, {config.n_targets} targets | "
    f"duration = {config.duration} | seed = {config.seed}\n"
    f"binding = {config.mechanics.binding_rate}, "
    f"unbinding = {config.mechanics.unbinding_rate} | "
    f"boundary = {config.domain.boundary}"
)

## Run the simulation

This cell runs one realization. The remaining cells use this stored `result`, so changing the animation or plots does not rerun the simulation. `result.snapshots`, `result.events`, and `result.final_cells` retain the recorded data; `result.config` retains the configuration used for this run.

In [ ]:
started = perf_counter()
result = simulate(config, progressbar=True)
elapsed = perf_counter() - started
counts = result.kills_per_cell()  # Includes killers with zero kills.

assert np.isclose(result.snapshots["time"].max(), result.duration)
assert len(result.final_cells) == result.config.n_killers + result.config.n_targets
assert counts.sum() == result.events["event"].eq("target_killed").sum()

clear_output(wait=True)
print(
    f"Finished in {elapsed:.2f} s; "
    f"{result.snapshots['step'].nunique():,} recorded snapshots."
)
display(pd.Series(result.summary(), name="value").to_frame())

## Watch the simulation

Use the playback controls to pause, advance frames, or replay. Blue cells are killers, green cells are living targets, and pink cells are dead targets. Trails show recent movement. The title shows simulation time; equal playback intervals can represent different time increments because integration is adaptive.

The preview displays at most `MAX_ANIMATION_FRAMES` frames, including the last snapshot. Increase this value for smoother playback and a larger notebook output. All recorded snapshots remain available for analysis. The options below follow the reference notebook's animation style.

In [ ]:
MAX_ANIMATION_FRAMES = 200
if MAX_ANIMATION_FRAMES < 2:
    raise ValueError("MAX_ANIMATION_FRAMES must be at least 2.")

recorded_frames = result.snapshots["step"].nunique()
frame_step = max(1, int(np.ceil((recorded_frames - 1) / (MAX_ANIMATION_FRAMES - 1))))
displayed_frames = len(range(0, recorded_frames, frame_step))
displayed_frames += (recorded_frames - 1) % frame_step != 0
print(f"Displaying {displayed_frames} of {recorded_frames:,} recorded frames.")

animation_params = {
    "killer_radius": result.config.killer_radius,
    "target_radius": result.config.target_radius,
    "dead_target_radius": result.config.target_radius,
    "killer_colour": "#92C4E9",
    "live_target_colour": "#96C36E",
    "dead_target_colour": "#EA9A9D",
    "cir_linewidth": 0.8,
    "traj_steps": 250,
    "traj_linewidth": 2.0,
    "traj_alpha": 0.45,
    "figsize": (6, 7),
    "fig_dpi": 100,
    "plot_margin": 0.06,
    "interval": 40,
    "frame_step": frame_step,
    "add_swatch_legend": True,
    "legend_labels": ["Killer", "Live target", "Dead target"],
    "render_html": True,
    "transparent_background": True,
}
with plt.rc_context({"animation.embed_limit": 100}):
    animation = animate(result, **animation_params)

## Inspect the outcomes

Deaths and activity are plotted against recorded simulation time. Thin activity lines show individual killers; the thick line shows their mean. Target damage is averaged over living targets, with a band spanning their minimum and maximum. This mean can fall when damaged targets die, even without recovery, and is undefined once no targets remain alive.

The kill distribution includes zero-count killers. Each death is credited to the killer assigned by the simulator's primary-damage attribution rule. These are counts from one realization; a flat curve or zero kills is a valid outcome.

In [ ]:
history = result.snapshots
killers = history.loc[history["cell_type"].eq("killer")]
targets = history.loc[history["cell_type"].eq("target")]
living_targets = targets.groupby("time")["alive"].sum().sort_index()
activity = killers.pivot(index="time", columns="cell_id", values="state").sort_index()
damage = (
    targets.loc[targets["alive"]]
    .groupby("time")["death_factor"].agg(["mean", "min", "max"])
    .reindex(living_targets.index)
)
frequencies = np.bincount(counts, minlength=1) / len(counts)
kill_numbers = np.arange(len(frequencies))

with plt.rc_context({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False}):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), dpi=120, layout="constrained")

    ax = axes[0, 0]
    ax.step(living_targets.index, result.config.n_targets - living_targets, where="post", color="#b05258")
    ax.set(title="Target deaths", xlabel="Time (model units)", ylabel="Dead targets",
           xlim=(0, result.duration), ylim=(0, result.config.n_targets))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    ax = axes[0, 1]
    ax.plot(activity.index, activity.to_numpy(), color="#92C4E9", alpha=0.35, linewidth=0.8)
    ax.plot(activity.index, activity.mean(axis=1), color="#24618f", linewidth=2, label="Mean")
    ax.set(title="Killer activity", xlabel="Time (model units)", ylabel="Killer state",
           xlim=(0, result.duration), ylim=(-0.02, 1.02))
    ax.legend(frameon=False)

    ax = axes[1, 0]
    ax.fill_between(damage.index, damage["min"], damage["max"], color="#96C36E", alpha=0.25, label="Min–max")
    ax.plot(damage.index, damage["mean"], color="#4b7938", linewidth=2, label="Mean")
    ax.axhline(result.config.cytotoxicity.death_threshold, color="0.3", linestyle="--", label="Death threshold")
    ax.set(title="Damage in living targets", xlabel="Time (model units)", ylabel="Damage",
           xlim=(0, result.duration), ylim=(0, 1.05 * result.config.cytotoxicity.death_threshold))
    ax.legend(frameon=False)

    ax = axes[1, 1]
    ax.bar(kill_numbers, frequencies, width=0.8, color="#85a993", edgecolor="black", linewidth=0.6)
    ax.set(title="Kills per killer", xlabel="Kills", ylabel="Proportion of killers",
           xlim=(-0.6, kill_numbers[-1] + 0.6), ylim=(0, 1))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.show()

## Per-killer counts

Compare contact episodes, formed synapses, kills, and final activity for every killer. Contacts and synapses count episodes, so repeated encounters with the same target are included.

In [ ]:
final_activity = (
    result.final_cells.loc[result.final_cells["cell_type"].eq("killer")]
    .set_index("cell_id")["state"]
    .reindex(range(result.config.n_killers))
)
per_killer = pd.DataFrame({
    "killer_id": np.arange(result.config.n_killers),
    "contacts": result.contacts_per_cell(),
    "synapses": result.synapses_per_cell(),
    "kills": counts,
    "final_activity": final_activity.to_numpy(),
})
display(per_killer)